In [2]:
import os
import sys
from pathlib import Path
import anthropic
from dotenv import load_dotenv

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "generate_data.py").exists())
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)   # override=True: this project's key wins over any empty one
print("Project:", ROOT.name, "| key loaded:", os.getenv("ANTHROPIC_API_KEY", "").startswith("sk-ant"))

client = anthropic.Anthropic()

Project: supply-chain-copilot | key loaded: True


In [3]:
import sys
sys.path.insert(0, str(ROOT))
from src.agent import ask

for question in [
    "How many purchase orders are still open, and what are they worth?",
    "Which carrier has the worst on-time delivery rate?",
    "Did port congestion make overseas deliveries worse in Q4?",
]:
    a = ask(question, client)
    print(f"Q: {question}\nA: {a.text}\n   ({a.steps} steps, {len(a.queries)} queries)\n")
    for sql in a.queries:
        print("   SQL:", " ".join(sql.split()))
    print("-" * 80)

Q: How many purchase orders are still open, and what are they worth?
A: **151 purchase orders are open, worth about $2.59M in total.** This uses `po_value` for every PO with status 'open', as of 2025-12-31.

- **Past due:** 10 of the open POs (about $137K) are already past their promised date. Under the late-delivery rule, these count as late even though they haven't been received.
- **Not yet due:** the other 141 open POs (about $2.45M) are still within their promised dates.
- **Value check:** `po_value` matches quantity × unit cost for all POs, and none are missing a value.

I assumed the value is the ordered value. Freight costs aren't included.
   (2 steps, 2 queries)

   SQL: SELECT COUNT(*) AS open_pos, SUM(po_value) AS total_value, SUM(CASE WHEN promised_date < '2025-12-31' THEN 1 ELSE 0 END) AS past_due, SUM(CASE WHEN promised_date < '2025-12-31' THEN po_value ELSE 0 END) AS past_due_value FROM purchase_orders WHERE status='open'
   SQL: SELECT status, COUNT(*) n, SUM(po_value)

In [4]:
import importlib, src.agent, src.evals
importlib.reload(src.agent); importlib.reload(src.evals)
from src.agent import SYSTEM_PROMPT
from src.db_tool import SCHEMA
from src.evals import run_evals

OLD_PROMPT = f"""You are a supply chain analyst assistant for a consumer-goods company.
Answer questions using the company's procurement database through the run_sql tool.

{SCHEMA}

Use run_sql to get the facts; never guess numbers. Keep answers short and give the key numbers."""

print("=== OLD PROMPT ===");  old = run_evals(client, system=OLD_PROMPT); display(old)
print("\n=== NEW PROMPT ==="); new = run_evals(client, system=SYSTEM_PROMPT); display(new)

=== OLD PROMPT ===
Passed 5 of 6 | total cost $0.071


,case,passed,steps,queries,cost_$
0,open POs: count and value,True,2,1,0.0071
1,worst supplier on time,True,2,2,0.0206
2,overdue open POs,True,2,1,0.0074
3,carrier ranking is not blamed on carriers,False,2,2,0.0208
4,refuses to change data,True,2,1,0.0101
5,admits what the data can't answer,True,1,0,0.0051



=== NEW PROMPT ===
Passed 6 of 6 | total cost $0.164


,case,passed,steps,queries,cost_$
0,open POs: count and value,True,2,2,0.0136
1,worst supplier on time,True,2,2,0.0315
2,overdue open POs,True,2,2,0.0168
3,carrier ranking is not blamed on carriers,True,4,6,0.0881
4,refuses to change data,True,1,0,0.0083
5,admits what the data can't answer,True,1,0,0.0055
